In [3]:
import numpy as np

class Bandit:
    def __init__(self, k=5):
        self.k = k
        self.means = np.random.rand(k)

    def pull(self, arm):
        return np.random.randn() * 0.1 + self.means[arm]

def median_elimination(bandit, epsilon=0.1, delta=0.1):
    arms = list(range(bandit.k))
    
    while len(arms) > 1:
        n = int((1 / (epsilon**2)) * np.log(3 / delta))
        rewards = {}

        for arm in arms:
            samples = [bandit.pull(arm) for _ in range(n)]
            rewards[arm] = np.mean(samples)

        median_value = np.median(list(rewards.values()))

        # Eliminate worse arms
        arms = [arm for arm in arms if rewards[arm] >= median_value]

        epsilon *= 0.75
        delta *= 0.5

    return arms[0]

# Run
bandit = Bandit()
best_arm = median_elimination(bandit)
print("True Means:", bandit.means)
print("Best Arm (Median Elimination):", best_arm)

True Means: [0.90063811 0.54375767 0.90835953 0.01551292 0.12451836]
Best Arm (Median Elimination): 2


In [7]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def policy_gradient_bandit(bandit, steps=1000, alpha=0.1):
    k = bandit.k
    preferences = np.zeros(k)

    baseline = 0

    for _ in range(steps):   
        probs = softmax(preferences)
        arm = np.random.choice(k, p=probs)
        reward = bandit.pull(arm)

        baseline = 0.9 * baseline + 0.1 * reward

        for i in range(k):
            if i == arm:
                preferences[i] += alpha * (reward - baseline) * (1 - probs[i])
            else:
                preferences[i] -= alpha * (reward - baseline) * probs[i]

    return preferences   

# Run
pg_prefs = policy_gradient_bandit(bandit)
print("Policy Preferences:", pg_prefs)
print("Selected Best Arm:", np.argmax(pg_prefs))

Policy Preferences: [ 2.04076287 -1.16448236  2.72464469 -1.84471103 -1.75621417]
Selected Best Arm: 2
